#Problem Statement & Objective

# Task 1: News Topic Classification using BERT

## Problem Statement
News articles belong to different categories such as World, Sports, Business, and Sci/Tech.
Manually categorizing large volumes of news headlines can be time-consuming and inefficient.

## Objective
The objective of this project is to build a transformer-based text classification model using BERT
to automatically classify news headlines into predefined categories.

## Dataset
AG News Dataset (Hugging Face)

## Categories
- World
- Sports
- Business
- Sci/Tech

Install Dependencies

In [4]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


Import Libraries

In [5]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import TrainingArguments, Trainer

from sklearn.metrics import accuracy_score, f1_score

Load Dataset

In [6]:
dataset = load_dataset("ag_news")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Dataset Exploration

In [7]:
dataset['train'][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [8]:
label_names = ["World", "Sports", "Business", "Sci/Tech"]

Load Tokenizer

In [9]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization Function

In [10]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

Apply Tokenization

In [11]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Convert to PyTorch Format

In [12]:
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Load BERT Model

In [13]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Configuration

In [14]:
training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    eval_strategy="epoch",
    weight_decay=0.01
)



Evaluation Metrics

In [15]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1_score": f1
    }

Trainer Setup

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(10000)),
    eval_dataset=tokenized_dataset["test"].shuffle(seed=42).select(range(2000)),
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)




Train Model

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,0.410144,0.282650,0.910000,0.909858


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=625, training_loss=0.38427846069335936, metrics={'train_runtime': 269.0465, 'train_samples_per_second': 37.168, 'train_steps_per_second': 2.323, 'total_flos': 657789450240000.0, 'train_loss': 0.38427846069335936, 'epoch': 1.0})

Model Evaluation

In [18]:
trainer.evaluate()

{'eval_loss': 0.2826501429080963,
 'eval_accuracy': 0.91,
 'eval_f1_score': 0.9098578165906428,
 'eval_runtime': 14.025,
 'eval_samples_per_second': 142.603,
 'eval_steps_per_second': 8.913,
 'epoch': 1.0}

Test Prediction

In [20]:
headline = "Apple launches new AI chip for smartphones"

# Tokenize the input
inputs = tokenizer(headline, return_tensors="pt")

# Move inputs to the same device as the model (e.g., GPU)
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get prediction
outputs = model(**inputs)
pred = outputs.logits.argmax().item()

print("Headline:", headline)
print("Predicted Category:", label_names[pred])

Headline: Apple launches new AI chip for smartphones
Predicted Category: Sci/Tech


Save Model

In [21]:
model.save_pretrained("bert_news_model")
tokenizer.save_pretrained("bert_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_news_model/tokenizer_config.json', 'bert_news_model/tokenizer.json')

## Conclusion

In this project, a BERT-based transformer model was fine-tuned to classify news headlines
into four categories: World, Sports, Business, and Sci/Tech.

The model achieved strong performance with high accuracy and F1-score on the AG News dataset.

This demonstrates how pre-trained transformer models can be effectively applied
to natural language processing tasks such as text classification.

In [22]:
!zip -r bert_news_model.zip bert_news_model

  adding: bert_news_model/ (stored 0%)
  adding: bert_news_model/tokenizer_config.json (deflated 42%)
  adding: bert_news_model/tokenizer.json (deflated 71%)
  adding: bert_news_model/model.safetensors (deflated 7%)
  adding: bert_news_model/config.json (deflated 55%)


In [23]:
from google.colab import files
files.download("bert_news_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.2 MB/s eta 0:00:00


In [25]:
%cd /content/Task1_BERT_News_Classifier

[Errno 2] No such file or directory: '/content/Task1_BERT_News_Classifier'
/content


In [26]:
!streamlit run app.py & npx localtunnel --port 8501

⠙Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py
⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://dirty-kings-try.loca.lt
^C


In [ ]:
y